In [5]:
import pandas as pd
import time
from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

INPUT_CSV = "fcp_input_urls.csv"
OUTPUT_CSV = "fitment_output.csv"
URL_COLUMN = "URL"

results = []

options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")

# Keep browser visible first. Add headless later if it works.
# options.add_argument("--headless=new")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

df = pd.read_csv(INPUT_CSV)

for url in tqdm(df[URL_COLUMN].dropna(), desc="Scraping FCP Euro Fitment"):
    try:
        driver.get(url)

        # Give page time to load
        time.sleep(5)

        # Scroll to help lazy-loaded fitment content appear
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        fitments = soup.select("div.fitmentGuide__applicationGroup li")

        if not fitments:
            results.append({
                "URL": url,
                "Application": "",
                "Details": "",
                "Notes": "",
                "Status": "No fitment rows found"
            })
            continue

        for fit in fitments:
            app_tag = fit.select_one(".fitmentGuide__application strong")
            details_tag = fit.select_one(".fitmentGuide__additional span")
            notes_tag = fit.select_one(".fitmentGuide__additional strong")

            application = app_tag.get_text(" ", strip=True) if app_tag else ""
            details = details_tag.get_text(" ", strip=True) if details_tag else ""
            notes = notes_tag.get_text(" ", strip=True).replace("Notes:", "").strip() if notes_tag else ""

            results.append({
                "URL": url,
                "Application": application,
                "Details": details,
                "Notes": notes,
                "Status": "Success"
            })

    except Exception as e:
        results.append({
            "URL": url,
            "Application": "",
            "Details": "",
            "Notes": "",
            "Status": f"Error: {e}"
        })

driver.quit()

output_df = pd.DataFrame(results)
output_df.to_csv(OUTPUT_CSV, index=False)

print(f"Done. Output saved to {OUTPUT_CSV}")

Scraping FCP Euro Fitment: 100%|██████████| 2/2 [00:18<00:00,  9.04s/it]


Done. Output saved to fitment_output.csv


403
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-OBvXdZyk03bD26SS61aYK5&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-s
